<!-- journey-badges -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NolaNexus/LMp2LMa/blob/main/track-03-ml-engineer/01_ensemble_modeling.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/NolaNexus/LMp2LMa/blob/main/track-03-ml-engineer/01_ensemble_modeling.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/NolaNexus/LMp2LMa)

# Ensemble Modeling: Stacking, Blending & Voting

**Module 2 - Notebook 1** | Skillsoft Track 3: ML Engineer

## Learning Objectives
- Implement voting ensembles (hard/soft), stacking, and blending
- Compare ensemble methods vs single models on classification & regression
- Build automated model selection framework with hyperparameter tuning
- Track experiments with MLflow for reproducibility
- Evaluate ensemble diversity and model complementarity

In [ ]:
# Setup
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# ML libraries
# MLflow for experiment tracking
import os

import catboost as cb
import lightgbm as lgb
import xgboost as xgb
from sklearn.datasets import make_classification, make_regression
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier,
    StackingRegressor,
    VotingClassifier,
    VotingRegressor,
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

# MLflow setup
mlflow.set_tracking_uri("sqlite:///mlruns_ensemble.db")
experiment_name = "ensemble_modeling"
experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment is None:
    experiment_id = mlflow.create_experiment(experiment_name)
else:
    experiment_id = experiment.experiment_id
mlflow.set_experiment(experiment_name)

sns.set_style("whitegrid")
np.random.seed(42)

print("MLflow tracking: sqlite:///mlruns_ensemble.db")
print(f"Experiment: {experiment_name} (ID: {experiment_id})")
print("Environment ready")

## 1. Voting Ensembles (Hard & Soft)

**VotingClassifier** combines predictions from multiple estimators:
- **Hard voting**: majority class label prediction
- **Soft voting**: average predicted class probabilities (requires `probability=True`)

In [ ]:
# Generate classification dataset
X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=15, n_redundant=5, random_state=42, class_sep=1.5
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Class distribution: {np.bincount(y_train)}")

In [ ]:
# Define base estimators
base_estimators = [
    ("lr", LogisticRegression(random_state=42, max_iter=1000, n_jobs=-1)),
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ("svc", SVC(probability=True, random_state=42)),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
    ("dt", DecisionTreeClassifier(random_state=42)),
]

# Hard voting
with mlflow.start_run(run_name="hard_voting") as run:
    hard_voting = VotingClassifier(estimators=base_estimators, voting="hard")
    hard_voting.fit(X_train, y_train)
    hard_pred = hard_voting.predict(X_test)
    hard_acc = accuracy_score(y_test, hard_pred)

    mlflow.log_param("voting_type", "hard")
    mlflow.log_metric("accuracy", hard_acc)
    mlflow.sklearn.log_model(
        hard_voting,
        "model",
        skops_trusted_types=[
            "sklearn.tree._tree.Tree",
            "sklearn.utils._bunch.Bunch",
            "sklearn.ensemble._forest.RandomForestRegressor",
            "sklearn.ensemble._gb.GradientBoostingRegressor",
        ],
    )

    print(f"Hard Voting Accuracy: {hard_acc:.4f}")
    run_id_hard = run.info.run_id

# Soft voting
with mlflow.start_run(run_name="soft_voting") as run:
    soft_voting = VotingClassifier(estimators=base_estimators, voting="soft")
    soft_voting.fit(X_train, y_train)
    soft_pred = soft_voting.predict(X_test)
    soft_acc = accuracy_score(y_test, soft_pred)
    soft_auc = roc_auc_score(y_test, soft_voting.predict_proba(X_test)[:, 1])

    mlflow.log_param("voting_type", "soft")
    mlflow.log_metric("accuracy", soft_acc)
    mlflow.log_metric("roc_auc", soft_auc)
    mlflow.sklearn.log_model(
        soft_voting,
        "model",
        skops_trusted_types=[
            "sklearn.tree._tree.Tree",
            "sklearn.utils._bunch.Bunch",
            "sklearn.ensemble._forest.RandomForestRegressor",
            "sklearn.ensemble._gb.GradientBoostingRegressor",
        ],
    )

    print(f"Soft Voting Accuracy: {soft_acc:.4f}")
    print(f"Soft Voting AUC: {soft_auc:.4f}")
    run_id_soft = run.info.run_id

# Individual model performance (baseline)
print("\n--- Individual Model Baselines ---")
individual_results = {}
for name, model in base_estimators:
    with mlflow.start_run(run_name=f"baseline_{name}"):
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        acc = accuracy_score(y_test, pred)
        prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
        auc = roc_auc_score(y_test, prob) if prob is not None else None

        mlflow.log_param("model_type", name)
        mlflow.log_metric("accuracy", acc)
        if auc:
            mlflow.log_metric("roc_auc", auc)

        individual_results[name] = {"accuracy": acc, "auc": auc}
        auc_str = f"{auc:.4f}" if auc is not None else "N/A"
        print(f"{name:20s}: Acc={acc:.4f}, AUC={auc_str}")

## 2. Stacking (Stacked Generalization)

**Stacking** uses a meta-learner to combine base estimator predictions:
- Base estimators generate out-of-fold predictions via cross-validation
- Meta-learner learns optimal combination of base predictions
- Typically outperforms simple voting when base models are diverse

In [ ]:
# Stacking with Logistic Regression meta-learner
with mlflow.start_run(run_name="stacking_lr_meta") as run:
    stacking = StackingClassifier(
        estimators=base_estimators,
        final_estimator=LogisticRegression(random_state=42, max_iter=1000),
        cv=5,
        stack_method="predict_proba",
        n_jobs=-1,
    )
    stacking.fit(X_train, y_train)
    stack_pred = stacking.predict(X_test)
    stack_acc = accuracy_score(y_test, stack_pred)
    stack_auc = roc_auc_score(y_test, stacking.predict_proba(X_test)[:, 1])

    mlflow.log_param("meta_learner", "LogisticRegression")
    mlflow.log_param("cv_folds", 5)
    mlflow.log_metric("accuracy", stack_acc)
    mlflow.log_metric("roc_auc", stack_auc)
    mlflow.sklearn.log_model(
        stacking,
        "model",
        skops_trusted_types=[
            "sklearn.tree._tree.Tree",
            "sklearn.utils._bunch.Bunch",
            "sklearn.ensemble._forest.RandomForestRegressor",
            "sklearn.ensemble._gb.GradientBoostingRegressor",
        ],
    )

    print(f"Stacking (LR meta) Accuracy: {stack_acc:.4f}")
    print(f"Stacking (LR meta) AUC: {stack_auc:.4f}")
    run_id_stack_lr = run.info.run_id

## 3. Gradient Boosting Libraries: XGBoost, LightGBM, CatBoost

Modern gradient boosting implementations with built-in regularization:
- **XGBoost**: Regularized gradient boosting with tree pruning
- **LightGBM**: Gradient-based one-side sampling, histogram-based splits
- **CatBoost**: Ordered boosting, handles categorical features natively

In [ ]:
# Train gradient boosting models
gb_models = {
    "XGBoost": xgb.XGBClassifier(
        random_state=42, eval_metric="logloss", n_estimators=200, learning_rate=0.1, max_depth=6
    ),
    "LightGBM": lgb.LGBMClassifier(
        random_state=42, n_estimators=200, verbose=-1, learning_rate=0.1, num_leaves=31
    ),
    "CatBoost": cb.CatBoostClassifier(
        random_state=42, iterations=200, verbose=False, learning_rate=0.1, depth=6
    ),
}

gb_results = {}
for name, model in gb_models.items():
    with mlflow.start_run(run_name=f"gb_{name}") as run:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        prob = model.predict_proba(X_test)[:, 1]
        acc = accuracy_score(y_test, pred)
        auc = roc_auc_score(y_test, prob)

        mlflow.log_param("model_type", name)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)
        mlflow.sklearn.log_model(
            model,
            "model",
            skops_trusted_types=[
                "sklearn.tree._tree.Tree",
                "sklearn.utils._bunch.Bunch",
                "sklearn.ensemble._forest.RandomForestRegressor",
                "sklearn.ensemble._gb.GradientBoostingRegressor",
            ],
        )

        gb_results[name] = {"accuracy": acc, "auc": auc}
        print(f"{name:15s}: Acc={acc:.4f}, AUC={auc:.4f}")

## 4. Stacking with Gradient Boosting as Meta-Learner

Using a powerful gradient boosting model as meta-learner often yields best results.

In [ ]:
# Stacking with XGBoost meta-learner
with mlflow.start_run(run_name="stacking_xgb_meta") as run:
    stacking_gb = StackingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
            ("et", ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
            ("gb", GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ("lr", LogisticRegression(random_state=42, max_iter=1000)),
        ],
        final_estimator=xgb.XGBClassifier(
            random_state=42, eval_metric="logloss", n_estimators=100, learning_rate=0.1, max_depth=4
        ),
        cv=5,
        stack_method="predict_proba",
        n_jobs=-1,
    )
    stacking_gb.fit(X_train, y_train)
    stack_gb_pred = stacking_gb.predict(X_test)
    stack_gb_acc = accuracy_score(y_test, stack_gb_pred)
    stack_gb_auc = roc_auc_score(y_test, stacking_gb.predict_proba(X_test)[:, 1])

    mlflow.log_param("meta_learner", "XGBoost")
    mlflow.log_metric("accuracy", stack_gb_acc)
    mlflow.log_metric("roc_auc", stack_gb_auc)
    mlflow.sklearn.log_model(
        stacking_gb,
        "model",
        skops_trusted_types=[
            "sklearn.tree._tree.Tree",
            "sklearn.utils._bunch.Bunch",
            "sklearn.ensemble._forest.RandomForestRegressor",
            "sklearn.ensemble._gb.GradientBoostingRegressor",
        ],
    )

    print(f"Stacking (XGB meta) Accuracy: {stack_gb_acc:.4f}")
    print(f"Stacking (XGB meta) AUC: {stack_gb_auc:.4f}")
    run_id_stack_gb = run.info.run_id

## 5. Model Selection Framework with Hyperparameter Tuning

Automated model selection with cross-validated hyperparameter search.

In [ ]:
import time

# Define models with parameter grids
model_configs = {
    "LogisticRegression": {
        "model": LogisticRegression(random_state=42, max_iter=1000, n_jobs=-1),
        "params": {"C": [0.1, 1.0, 10.0], "penalty": ["l2"]},
    },
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42, n_jobs=-1),
        "params": {"n_estimators": [100, 200], "max_depth": [None, 10, 20]},
    },
    "XGBoost": {
        "model": xgb.XGBClassifier(random_state=42, eval_metric="logloss"),
        "params": {"n_estimators": [100, 200], "max_depth": [3, 6], "learning_rate": [0.01, 0.1]},
    },
    "LightGBM": {
        "model": lgb.LGBMClassifier(random_state=42, verbose=-1),
        "params": {
            "n_estimators": [100, 200],
            "num_leaves": [31, 63],
            "learning_rate": [0.01, 0.1],
        },
    },
    "CatBoost": {
        "model": cb.CatBoostClassifier(random_state=42, verbose=False),
        "params": {"iterations": [100, 200], "depth": [4, 6], "learning_rate": [0.01, 0.1]},
    },
}

# Non-tuned ensembles
ensemble_models = {
    "Voting (soft)": VotingClassifier(estimators=base_estimators, voting="soft"),
    "Stacking (LR meta)": StackingClassifier(
        estimators=base_estimators,
        final_estimator=LogisticRegression(random_state=42, max_iter=1000),
        cv=5,
        stack_method="predict_proba",
        n_jobs=-1,
    ),
    "Stacking (XGB meta)": StackingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
            ("et", ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
            ("gb", GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ("lr", LogisticRegression(random_state=42, max_iter=1000)),
        ],
        final_estimator=xgb.XGBClassifier(
            random_state=42, eval_metric="logloss", n_estimators=100, learning_rate=0.1, max_depth=4
        ),
        cv=5,
        stack_method="predict_proba",
        n_jobs=-1,
    ),
}

In [ ]:
# Model selection with GridSearchCV + MLflow tracking
results = {}

print("Starting model selection with hyperparameter tuning...")
print("=" * 60)

# Tune individual models
for name, config in model_configs.items():
    start = time.time()
    with mlflow.start_run(run_name=f"tuned_{name}") as run:
        search = GridSearchCV(
            config["model"], config["params"], cv=3, scoring="roc_auc", n_jobs=-1, verbose=0
        )
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        best_params = search.best_params_

        pred = best_model.predict(X_test)
        prob = (
            best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, "predict_proba") else pred
        )
        acc = accuracy_score(y_test, pred)
        auc = roc_auc_score(y_test, prob)

        # Log to MLflow
        for param, value in best_params.items():
            mlflow.log_param(param, value)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)
        mlflow.log_metric("cv_best_score", search.best_score_)
        mlflow.log_metric("train_time_sec", time.time() - start)
        mlflow.sklearn.log_model(
            best_model,
            "model",
            skops_trusted_types=[
                "sklearn.tree._tree.Tree",
                "sklearn.utils._bunch.Bunch",
                "sklearn.ensemble._forest.RandomForestRegressor",
                "sklearn.ensemble._gb.GradientBoostingRegressor",
            ],
        )

        results[name] = {
            "accuracy": acc,
            "auc": auc,
            "time": time.time() - start,
            "params": best_params,
        }
        print(f"{name:20s}: Acc={acc:.4f}, AUC={auc:.4f}, Time={time.time()-start:.1f}s")

# Evaluate ensemble models (no tuning)
for name, model in ensemble_models.items():
    start = time.time()
    with mlflow.start_run(run_name=f"ensemble_{name}") as run:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else pred
        acc = accuracy_score(y_test, pred)
        auc = roc_auc_score(y_test, prob) if prob is not None else None

        mlflow.log_metric("accuracy", acc)
        if auc:
            mlflow.log_metric("roc_auc", auc)
        mlflow.log_metric("train_time_sec", time.time() - start)
        mlflow.sklearn.log_model(
            model,
            "model",
            skops_trusted_types=[
                "sklearn.tree._tree.Tree",
                "sklearn.utils._bunch.Bunch",
                "sklearn.ensemble._forest.RandomForestRegressor",
                "sklearn.ensemble._gb.GradientBoostingRegressor",
            ],
        )

        results[name] = {
            "accuracy": acc,
            "auc": auc,
            "time": time.time() - start,
            "params": "ensemble",
        }
        print(
            f"{name:20s}: Acc={acc:.4f}, AUC={auc:.4f if auc else 'N/A'}, Time={time.time()-start:.1f}s"
        )

In [ ]:
# Results summary
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values("auc", ascending=False)

print("\n" + "=" * 60)
print("MODEL SELECTION RESULTS (sorted by AUC)")
print("=" * 60)
print(results_df[["accuracy", "auc", "time"]].to_string())

# Best model
best_model_name = results_df.index[0]
print(f"\n🏆 Best Model: {best_model_name}")
print(f"   Accuracy: {results_df.loc[best_model_name, 'accuracy']:.4f}")
print(f"   AUC: {results_df.loc[best_model_name, 'auc']:.4f}")
print(f"   Params: {results_df.loc[best_model_name, 'params']}")

## 6. Regression Ensembles

Ensemble methods work equally well for regression tasks:
- **VotingRegressor**: average predictions from multiple regressors
- **StackingRegressor**: meta-regressor learns optimal combination
- **Bagging/Boosting**: variance reduction through aggregation

In [ ]:
# Generate regression dataset
X_reg, y_reg = make_regression(
    n_samples=2000, n_features=20, n_informative=15, noise=0.1, random_state=42
)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f"Regression Train: {X_train_r.shape}, Test: {X_test_r.shape}")

# Base regressors
reg_estimators = [
    ("ridge", Ridge(alpha=1.0)),
    (
        "rf",
        RandomForestClassifier(n_estimators=100, random_state=42),
    ),  # RF also works for regression
    ("gb", GradientBoostingClassifier(n_estimators=100, random_state=42)),
]

# Need proper regressors
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression

reg_estimators = [
    ("ridge", Ridge(alpha=1.0)),
    ("rf", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
    ("gb", GradientBoostingRegressor(n_estimators=100, random_state=42)),
    ("lr", LinearRegression()),
]

# Voting Regressor
with mlflow.start_run(run_name="voting_regressor") as run:
    voting_reg = VotingRegressor(estimators=reg_estimators)
    voting_reg.fit(X_train_r, y_train_r)
    pred = voting_reg.predict(X_test_r)
    mse = mean_squared_error(y_test_r, pred)
    r2 = r2_score(y_test_r, pred)

    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(
        voting_reg,
        "model",
        skops_trusted_types=[
            "sklearn.tree._tree.Tree",
            "sklearn.utils._bunch.Bunch",
            "sklearn.ensemble._forest.RandomForestRegressor",
            "sklearn.ensemble._gb.GradientBoostingRegressor",
        ],
    )

    print(f"Voting Regressor: MSE={mse:.4f}, R2={r2:.4f}")

# Stacking Regressor
with mlflow.start_run(run_name="stacking_regressor") as run:
    stacking_reg = StackingRegressor(
        estimators=reg_estimators, final_estimator=Ridge(alpha=1.0), cv=5, n_jobs=-1
    )
    stacking_reg.fit(X_train_r, y_train_r)
    pred = stacking_reg.predict(X_test_r)
    mse = mean_squared_error(y_test_r, pred)
    r2 = r2_score(y_test_r, pred)

    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(
        stacking_reg,
        "model",
        skops_trusted_types=[
            "sklearn.tree._tree.Tree",
            "sklearn.utils._bunch.Bunch",
            "sklearn.ensemble._forest.RandomForestRegressor",
            "sklearn.ensemble._gb.GradientBoostingRegressor",
        ],
    )

    print(f"Stacking Regressor: MSE={mse:.4f}, R2={r2:.4f}")

## 7. Ensemble Diversity Analysis

Ensemble effectiveness depends on base model diversity:
- **Correlation**: low prediction correlation between base models → better ensemble
- **Disagreement**: models should make different errors
- **Q-statistic**: measures pairwise diversity

In [ ]:
# Ensemble diversity analysis
base_models = [model for _, model in base_estimators]
base_models = [m.fit(X_train, y_train) for m in base_models]

# Get predictions from each base model
predictions = np.column_stack([m.predict(X_test) for m in base_models])

# Pairwise correlation matrix
corr_matrix = np.corrcoef(predictions.T)

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    xticklabels=[name for name, _ in base_estimators],
    yticklabels=[name for name, _ in base_estimators],
)
plt.title("Base Model Prediction Correlations (Lower = More Diverse)")
plt.tight_layout()
plt.show()

# Average pairwise correlation
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
avg_corr = corr_matrix[mask].mean()
print(f"Average pairwise correlation: {avg_corr:.3f}")
print("(Lower correlation → more diverse ensemble → better ensemble performance)")

## 8. Results Visualization & MLflow Summary

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
models = list(results_df.index)
accs = results_df["accuracy"].values
aucs = results_df["auc"].values

axes[0].barh(models, accs, color="steelblue", alpha=0.7)
axes[0].set_xlabel("Accuracy")
axes[0].set_title("Model Accuracy Comparison")
axes[0].set_xlim(0.5, 1.0)

axes[1].barh(models, aucs, color="coral", alpha=0.7)
axes[1].set_xlabel("ROC AUC")
axes[1].set_title("Model ROC AUC Comparison")
axes[1].set_xlim(0.5, 1.0)

plt.tight_layout()
plt.show()

In [ ]:
# MLflow experiment summary
client = MlflowClient()
runs = client.search_runs(experiment_ids=[experiment_id], max_results=50)

print(f"Total runs in experiment: {len(runs)}")
print("\nRun Summary:")
print(f"{'Run Name':<30} {'Model Type':<20} {'Accuracy':>10} {'AUC':>10} {'Time(s)':>10}")
print("-" * 80)

for run in sorted(runs, key=lambda r: r.data.metrics.get("roc_auc", 0), reverse=True):
    name = run.data.tags.get("mlflow.runName", "unnamed")
    model = run.data.params.get("model_type", run.data.params.get("meta_learner", "ensemble"))
    acc = run.data.metrics.get("accuracy", 0)
    auc = run.data.metrics.get("roc_auc", 0)
    t = run.data.metrics.get("train_time_sec", 0)
    print(f"{name:<30} {model:<20} {acc:>10.4f} {auc:>10.4f} {t:>10.1f}")

## Summary

| Ensemble Method | Key Insight | Best Use Case |
|-----------------|-------------|---------------|
| **Hard Voting** | Simple majority vote | Fast, interpretable, good baseline |
| **Soft Voting** | Probability averaging | Better calibrated probabilities |
| **Stacking** | Meta-learner optimizes combination | Best performance, diverse base models |
| **GB Meta-learner** | Powerful meta-learner | Best overall performance |
| **Voting Regressor** | Average predictions | Robust regression baseline |
| **Stacking Regressor** | Meta-regressor | Best regression performance |

**Key Takeaways**:
1. **Diversity matters**: Low correlation between base models → better ensemble
2. **Stacking > Voting**: Meta-learner learns optimal combinations
3. **GB meta-learners win**: XGBoost/LightGBM as meta-learners often win
4. **Track with MLflow**: Compare all variants, promote best to registry

**Next Steps**:
- Try blending (hold-out meta-training set)
- Explore dynamic ensemble selection
- Implement ensemble pruning for inference speed
- Try multi-level stacking

## MLflow UI Access

To view the experiment tracking UI, run in terminal:

```bash
mlflow ui --backend-store-uri sqlite:///mlruns_ensemble.db --port 5000
```

Then open http://localhost:5000 in your browser.